# Generación de una base de datos mysql con los datos importados

In [ ]:
%pip install sqlalchemy cryptography pymysql 'git+https://github.com/dsevilla/jupysql' 'pandas[performance,parquet]'

In [ ]:
%load_ext sql
%config SqlMagic.autopandas=True
%config SqlMagic.displaycon=False
%config SqlMagic.named_parameters="enabled"

In [ ]:
import pandas as pd

In [ ]:
db_hostname: str = "localhost"

In [ ]:
DATABASE_URL: str=f"mysql+pymysql://root:root@{db_hostname}/?charset=utf8mb4&collation=utf8mb4_bin"

In [ ]:
%env DATABASE_URL=$DATABASE_URL

In [ ]:
from sqlalchemy import Engine, create_engine

engine: Engine = create_engine(DATABASE_URL,
                               isolation_level="READ COMMITTED")

In [ ]:
%%sql engine
-- Configuración de la base de datos
SET GLOBAL innodb_redo_log_capacity = 1000 * 1024 * 1024;
SET NAMES utf8mb4 COLLATE utf8mb4_bin;

In [ ]:
%%sql
DROP DATABASE IF EXISTS stackoverflow;
CREATE DATABASE stackoverflow CHARACTER SET utf8mb4 COLLATE utf8mb4_bin;

In [ ]:
%%sql
USE stackoverflow;

Definimos una función para leer el esquema de cada conjunto de datos. Lo describimos anteriormente como un esquema PyArrow.


In [ ]:
import pyarrow as pa
from pyarrow import Schema
import requests
import io

def read_remote_parquet_schema(url: str) -> Schema:
    """
    Downloads a remote Parquet file and returns its PyArrow schema.
    Args:
        url (str): The URL to the Parquet file.
    Returns:
        pyarrow.Schema: The schema of the Parquet file.
    """
    response: requests.Response = requests.get(url)
    response.raise_for_status()
    buffer = io.BytesIO(response.content)
    return pa.ipc.read_schema(buffer)

In [ ]:
# Read the schema directly from the URL
BASE_URL: str = "https://github.com/dsevilla/bd2-data/raw/refs/heads/main/es.stackoverflow/parquet/"

posts_schema: pa.Schema = \
    read_remote_parquet_schema(f"{BASE_URL}/Posts.pb")
users_schema: pa.Schema = \
    read_remote_parquet_schema(f"{BASE_URL}/Users.pb")
tags_schema: pa.Schema = \
    read_remote_parquet_schema(f"{BASE_URL}/Tags.pb")
comments_schema: pa.Schema = \
    read_remote_parquet_schema(f"{BASE_URL}/Comments.pb")
votes_schema: pa.Schema = \
    read_remote_parquet_schema(f"{BASE_URL}/Votes.pb")

In [ ]:
def gen_create_table_from_schema(schema: pa.Schema, table_name: str) -> str:
    """
    Generate a CREATE TABLE SQL statement from a PyArrow schema.

    Args:
        schema: PyArrow schema
        table_name: Name for the SQL table

    Returns:
        SQL CREATE TABLE statement as string

    Metadata support:
        - 'primary_key': 'true' - marks column as primary key
        - 'foreign_key': 'Table.Attribute' - creates foreign key reference
    """
    # PyArrow to SQL type mapping
    type_mapping: dict[pa.DataType, str] = {
        pa.int8(): "TINYINT",
        pa.int16(): "SMALLINT",
        pa.int32(): "INTEGER",
        pa.int64(): "BIGINT",
        pa.uint8(): "TINYINT UNSIGNED",
        pa.uint16(): "SMALLINT UNSIGNED",
        pa.uint32(): "INTEGER UNSIGNED",
        pa.uint64(): "BIGINT UNSIGNED",
        pa.float32(): "REAL",
        pa.float64(): "DOUBLE",
        pa.bool_(): "BOOLEAN",
        pa.string(): "TEXT",
        pa.large_string(): "TEXT",
        pa.binary(): "BLOB",
        pa.large_binary(): "BLOB",
    }

    def get_sql_type(pa_type: pa.DataType) -> str:
        # Handle timestamp types
        if pa.types.is_timestamp(pa_type):
            return "TIMESTAMP"
        elif pa.types.is_date(pa_type):
            return "DATE"
        elif pa.types.is_time(pa_type):
            return "TIME"
        elif pa.types.is_decimal(pa_type):
            return f"DECIMAL({pa_type.precision},{pa_type.scale})"
        elif pa.types.is_string(pa_type) or pa.types.is_large_string(pa_type):
            return "TEXT"
        # Direct mapping lookup
        return type_mapping.get(pa_type, "TEXT")

    # Helper to build DEFAULT clause
    def build_default_clause(fld: pa.Field) -> str:
        """Return the SQL DEFAULT clause (including leading space) for a field or
        an empty string when no default is present or it cannot be parsed.

        Args:
            fld: pyarrow Field instance

        Returns:
            A string like " DEFAULT 0" or " DEFAULT 'abc'" or empty string.
        """

        if not (fld.metadata and b"default" in fld.metadata):
            return ""

        default_val = fld.metadata[b"default"]
        try:
            # Integer types: treat binary 0 as 0
            if pa.types.is_integer(fld.type) or pa.types.is_unsigned_integer(fld.type):
                return f" DEFAULT {int(default_val.decode('utf-8'))}"
            elif pa.types.is_floating(fld.type):
                return f" DEFAULT {float(default_val.decode('utf-8'))}"
            elif pa.types.is_boolean(fld.type):
                sval = default_val.decode("utf-8").lower()
                return " DEFAULT 1" if sval in ("1", "true", "yes") else " DEFAULT 0"
            else:
                # treat as string-like
                sval = default_val.decode("utf-8")
                # escape single quotes
                sval = sval.replace("'", "''")
                return f" DEFAULT '{sval}'"
        except Exception:
            # fallback: ignore default on error
            return ""

    # Generate column definitions
    columns: list[str] = []
    primary_keys: list[str] = []
    foreign_keys: list[str] = []

    for field in schema:
        col_name: str = field.name
        sql_type: str = get_sql_type(field.type)

        # Check for nullable
        null_constraint: str = "" if field.nullable else " NOT NULL"

        # Build default clause using helper
        default_clause = build_default_clause(field)

        column_def: str = f"`{col_name}` {sql_type}{null_constraint}{default_clause}"
        columns.append(column_def)

        # Check metadata for primary key
        if field.metadata and b"primary_key" in field.metadata:
            if field.metadata[b"primary_key"] == b"true":
                primary_keys.append(col_name)

        # Check metadata for foreign key
        if field.metadata and b"foreign_key" in field.metadata:
            ref_table_col: str = field.metadata[b"foreign_key"].decode("utf-8")
            # Convert Table.Attribute to Table(Attribute)
            if "." in ref_table_col:
                table, attr = ref_table_col.split(".", 1)
                ref_table_col_sql = f"{table}({attr})"
            else:
                ref_table_col_sql = ref_table_col
            foreign_keys.append(f"FOREIGN KEY (`{col_name}`) REFERENCES {ref_table_col_sql}")

    # Build the CREATE TABLE statement
    create_parts: list[str] = [f"CREATE TABLE `{table_name}` ("]
    create_parts.append("    " + ",\n    ".join(columns))

    # Add primary key constraint
    if primary_keys:
        pk_cols: str = ", ".join(f"`{pk}`" for pk in primary_keys)
        create_parts.append(f",\n    PRIMARY KEY ({pk_cols})")

    # Add foreign key constraints
    if foreign_keys:
        for fk in foreign_keys:
            create_parts.append(f",\n    {fk}")

    create_parts.append(")\nCHARACTER SET utf8mb4 COLLATE utf8mb4_bin;")

    return "\n".join(create_parts)

## Importación de los datos

Eliminar temporalmente la comprobación de la consistencia de claves ajenas. Esto se hace por varias razones:

1. Para permitir la eliminación o modificación de registros en tablas que tienen claves foráneas sin que se produzcan errores de restricción.
2. Para facilitar la carga masiva de datos en tablas relacionadas sin tener que preocuparse por las restricciones de integridad referencial.
3. Para realizar operaciones de mantenimiento en la base de datos que pueden requerir la eliminación temporal de restricciones.

Es importante recordar que desactivar las comprobaciones de claves ajenas puede llevar a inconsistencias en los datos, por lo que se debe hacer con precaución y asegurarse de volver a habilitarlas una vez que se haya completado la operación necesaria.

In [ ]:
%%sql
SET GLOBAL foreign_key_checks = 0;

Empiezo la descarga de los ficheros parquet y preparamos el entorno para la ejecución de tareas en segundo plano.

In [ ]:
from pandas import DataFrame

# Mock of asyncio.Queue that prints output
class Queue:
    def put_nowait(self, item):
        print(f"Log: {item}")

def load_dataframe(tablename: str, parquetfile_or_files: str | list[str], logqueue: Queue = Queue()) -> bool:
    # Si nos envían na lista, concatenamos los archivos
    match parquetfile_or_files:
        case str():
            df: DataFrame = pd.read_parquet(f"{BASE_URL}/{parquetfile_or_files}")
        case list():
            df: DataFrame = pd.concat([pd.read_parquet(f"{BASE_URL}/{f}") for f in parquetfile_or_files])
    logqueue.put_nowait(f"Loaded {len(df)} rows from {tablename} dataframe.") if logqueue else None

    df.set_index('Id', inplace=True)

    try:
        result: int | None = df.to_sql(tablename, con=engine, if_exists='append',
                  index=True, schema='stackoverflow',
                        chunksize=1000,
                        method='multi')

        logqueue.put_nowait(f"Loaded {tablename} Dataframe. Result: {result}.") if logqueue else None

    except Exception as e:
        logqueue.put_nowait(f"Error loading {tablename} Dataframe: {e}.") if logqueue else None
        return False
    finally:
        del df
    return True

In [ ]:
users_schema_sql: str = \
    gen_create_table_from_schema(users_schema, 'Users')

print(users_schema_sql)

In [ ]:
%%sql
{{users_schema_sql}}

In [ ]:
load_dataframe('Users', 'Users.parquet')

In [ ]:
posts_schema_sql: str = \
    gen_create_table_from_schema(posts_schema, 'Posts')

print(posts_schema_sql)

In [ ]:
%%sql
{{posts_schema_sql}}

In [ ]:
load_dataframe('Posts', ['Posts1.parquet', 'Posts2.parquet', 'Posts3.parquet'])

In [ ]:
tags_schema_sql: str = \
    gen_create_table_from_schema(tags_schema, 'Tags')

print(tags_schema_sql)

In [ ]:
%%sql
{{tags_schema_sql}}

In [ ]:
load_dataframe('Tags', 'Tags.parquet')

In [ ]:
comments_schema_sql: str =\
    gen_create_table_from_schema(comments_schema, 'Comments')

print(comments_schema_sql)

In [ ]:
%%sql
{{comments_schema_sql}}

In [ ]:
load_dataframe('Comments', 'Comments.parquet')

In [ ]:
votes_schema_sql: str =\
    gen_create_table_from_schema(votes_schema, 'Votes')

print(votes_schema_sql)

In [ ]:
%%sql
{{votes_schema_sql}}

In [ ]:
load_dataframe('Votes', 'Votes.parquet')

In [ ]:
%%sql
DELETE FROM Votes WHERE PostId NOT IN (SELECT Id FROM Posts);

Ahora, vuelvo a reestablecer las comprobaciones de clave ajena porque después haremos un ejercicio relacionado.

In [ ]:
%%sql
SET GLOBAL foreign_key_checks = 1;

In [ ]:
%%sql
-- Y ahora sí
ALTER TABLE Votes ADD FOREIGN KEY (PostId) REFERENCES Posts(Id),
                  ADD FOREIGN KEY (UserId) REFERENCES Users(Id);

In [ ]:
%%sql
SHOW TABLES;

In [ ]:
%%sql
OPTIMIZE TABLE Posts;
OPTIMIZE TABLE Users;
OPTIMIZE TABLE Tags;
OPTIMIZE TABLE Comments;
OPTIMIZE TABLE Votes;

In [ ]:
%%sql
SHOW BINARY LOGS;

In [ ]:
%%sql
PURGE BINARY LOGS BEFORE NOW();
RESET MASTER;